# Streamlined Building Unit Prediction
Generalizing building unit regression calculations so they can be reproducible across all California data.

### Geospatial Data Sources:
- Building parquet file ([Google Earth Engine](https://sat-io.earthengine.app/view/gba))
- Zillow points (from advisor – not public)
- Residential parcels ([ArcGIS Hub, County of Los Angeles](https://hub.arcgis.com/documents/baaf8251bfb94d3984fb58cb5fd93258/about))

For a more descriptive overview of this process, see the [original notebook](https://github.com/sofiasarak/building-unit-regression/blob/main/unit_regression_la.ipynb).

In [1]:
# import necessary libraries
import pandas as pd
from shapely.geometry import box
import numpy as np
import geopandas as gpd
import os
import matplotlib.pyplot as plt
import fiona
import pandas as pd

# statistical libraries
#import sys
#!{sys.executable} -m pip install statsmodels
import statsmodels.formula.api as smf
# from sklearn.linear_model import LinearRegression

In [ ]:
# os.environ['PROJ_LIB'] = '/opt/anaconda3/share/proj'

In [2]:
# set option to see all data frame columns
pd.set_option('display.max_columns', None)

## Loading in Data

### Parcel

In [4]:
# load for small section of PGE counties
# load parcels only for PGE counties
parcels = gpd.read_file(
    "/../../capstone/electrigrid/data/parcels/Parcels_CA_2014.gdb",
    layer="CA_PARCELS_STATEWIDE",
    where="County='Ventura'").to_crs(epsg=4326)

## Zillow

In [5]:
# import Zillow data (make take 10-20 minutes)
zillow = gpd.read_file("/../../capstone/electrigrid/data/buildings/final_zillow.gpkg").to_crs(epsg=4326)

## Building Footprints

In [6]:
# import building footprint as geopandas dataframe (may take 1-5 minutes)
fp = "../../../../../capstone/electrigrid/data/buildings/buildings_ca.parquet"
building = gpd.read_parquet(fp).to_crs(epsg=4326)

# Building Unit Prediction Analysis

Reminders:
- `zillow_multi` represents the Zillow points (inaccurate), whereas 'multi_complete' comes from the actual buildings
- the Zillow and buildings dataframes contains data for ALL of California, whereas parcels are county-specific (for right now)

## Building Linear Model

The model will be trained using multi-family homes that directly intersect with Zillow points.

In [8]:
## FINDING ALL MULTI-FAMILY BUILDINGS BY JOINING ZILLOW -> PARCEL, THEN PARCEL -> BUILDINGS

# select only multi-family data from Zillow
zillow_multi = zillow[zillow['type'] == "Multi"]
zillow_multi = zillow_multi[zillow_multi['code'] != "RR106"]

## crop only to residential parcels
# keep the indices where multi-family homes match to parcels (.index.unique() de-duplicates)
valid_parcels = parcels.sjoin(zillow_multi, how = "inner", predicate="intersects")[parcels.columns].index.unique()

# select the parcels that match these indices
parcels_res = parcels.loc[valid_parcels]

# confirm that joining with Zillow decreases the number of parcels
# assert len(parcels_res) < len(parcels)

# crop to residential buildings (by keeping only those within residential parcels)
valid_buildings = building.sjoin(parcels_res, predicate="intersects").index.unique()
buildings_res = building.loc[valid_buildings]

# confirm that joining with Zillow decreased the number of buildings
assert len(buildings_res) < len(building)

## CALCULATING VOLUME FOR BOTH SINGLE AND MULTI-FAMILY HOMES

# keep all residential buildings, and add zillow points only where they match up (MULTI)
building_zillow_multi = gpd.sjoin(
    buildings_res,
    zillow_multi,
    how = "left",
    predicate = "intersects")

# filter for only single and condo units
zillow_single = zillow[(zillow['type'] == "Single") | (zillow['code'] == "RR106")]

# keep all residential buildings, and add zillow points only where they match up (SINGLE & CONDOS)
building_zillow_single = gpd.sjoin(
    building,
    zillow_single,
    how = "inner",
    predicate = "intersects")

# combine all for volume calculation
building_zillow_all = pd.concat([building_zillow_multi, building_zillow_single])

# reproject data frame to crs with meters as units
building_m = building_zillow_all.to_crs("EPSG:6933")

# create column from polygon area
building_m['area_m2'] = building_m.geometry.area

# rename height column to be clear about units
building_m.rename(columns={"height":"height_m"}, inplace = True)

# create volume column
building_m['volume_m3'] = building_m['area_m2'] * building_m['height_m']

# save single and condos as its own df
non_multi = building_m[(building_m['type'] == "Single") | (building_m['code'] == "RR106")]

# now that single unit homes also have volume data, we can drop them
building_m = building_m[building_m['type'] == "Multi"]
building_m = building_m[building_m['code'] != "RR106"]

## BUILDING REGRESSION TO PREDICT UNIT DATA WHERE IT IS MISSING

# keep only observations with unit data
building_w_units = building_m[~building_m['unit'].isna()]

assert building_w_units['unit'].isna().sum() == 0

# run model
results = smf.ols('unit ~ volume_m3', data=building_w_units).fit()

# add residuals as a column
building_w_units['residual'] = results.resid.copy()

# keep only observations that are less/equal to 2 standard deviations from residuals
building_units_clean = building_w_units[building_w_units['residual'].abs() <= 2 * building_w_units['residual'].std()]

# save outliers, as we will re-predict them using the regression
building_outliers = building_w_units[building_w_units['residual'].abs() > 2 * building_w_units['residual'].std()]

# rerun linear regression
results_clean = smf.ols('unit ~ volume_m3', data=building_units_clean).fit()

# save variables
intercept = results_clean.params[0]
slope = results_clean.params[1]

ValueError: zero-size array to reduction operation maximum which has no identity

## Predicting Units for Multi-Family Homes

We will select all residential multi-family homes by joining building footprints (`buildings_res`) and multi-family Zillow data (`zillow_multi`) on the parcel column. Then, the unit column will be dropped, to ensure that the number of units is not repeated for every building in the parcel. New unit information will be predicted using our model.

In [38]:
# join zillow point to parcels
zillow_w_par = zillow_multi.sjoin(parcels_res, predicate="within")

# join buildings to parcels
building_w_par = buildings_res.sjoin(parcels_res, predicate="within")

# join the two based on parcel number
building_w_zillow = pd.merge(zillow_w_par, building_w_par, on = 'PARNO')
building_w_zillow = building_w_zillow.drop('unit', axis = 1)

# clean columns
building_w_zillow = building_w_zillow.loc[:, "type":"code"].join(building_w_zillow.loc[:, "id":"geometry_y"])

# to gdf
building_w_zillow = gpd.GeoDataFrame(building_w_zillow, geometry="geometry_y", crs="EPSG:4326")

In [43]:
# calculate volume
# reproject data frame to crs with meters as units
building_w_zillow_m = building_w_zillow.to_crs("EPSG:6933")

# create column from polygon area
building_w_zillow_m['area_m2'] = building_w_zillow_m.geometry.area

# rename height column to be clear about units
building_w_zillow_m.rename(columns={"height":"height_m"}, inplace = True)

# create volume column
building_w_zillow_m['volume_m3'] = building_w_zillow_m['area_m2'] * building_w_zillow_m['height_m']

In [44]:
building_w_units['residual']

1931048    -3.584490
1931049    -4.421438
1936472    30.602684
1936493    -2.691220
1936494    -0.883883
             ...    
3714068    -1.179159
3714197    -3.605413
3714199    -3.864319
3714204   -10.181951
3714205    -5.797036
Name: residual, Length: 189, dtype: float64

In [46]:
# predict units based on volume
building_w_zillow_m['unit'] = round(intercept + building_w_zillow_m['volume_m3'] * slope)

## UNIT PREDICTION COMPLETE

# find centroid of footprints
building_w_zillow_m['centroid'] = building_w_zillow_m.geometry.centroid

# set as geometry
building_w_zillow_n = gpd.GeoDataFrame(building_w_zillow_m, geometry="centroid", crs="EPSG:4326")

## Concatenating with Single-Family Homes

Now that both our single and multi family homes contain points as their geomtries, we can combine them into one, and save.

In [53]:
# pre-join cleaning
non_multi = non_multi.to_crs(zillow.crs).drop('index_right', axis = 1)

# join Zillow data to non-multi family homes (takes ~1 minute)
non_multi_points = gpd.sjoin(
    zillow, # left df's geometry is always kept
    non_multi,
    how = "inner",
    predicate = "intersects")

# this join introduces multi-family homes that overlap single-family points
non_multi_points = non_multi_points[non_multi_points['type'] != "Multi"]

In [64]:
# selecting only necessary columns
multi = building_w_zillow_m[['type', 'year', 'room', 'heat', 'cool', 'own', 'value', 'sqft_type', 'ID', 'GEOID', 'p_ID', 'code', 'area_m2', 'volume_m3', 'unit', 'centroid']]
multi = multi.rename(columns = {"centroid":"geometry"})

# renaming
non_multi_points.columns = non_multi_points.columns.str.replace("_left", "")
single = non_multi_points[['type', 'year', 'room', 'heat', 'cool', 'own', 'value', 'sqft_type', 'ID', 'GEOID', 'p_ID', 'code', 'area_m2', 'volume_m3', 'unit', 'geometry']]

In [75]:
# concatenate multi and single family homes
ventura_building = pd.concat([single, multi]).to_crs(zillow.crs)

In [ ]:
ventura_building.to_file("data/ventura_buildings.geojson", driver='GeoJSON')

In [115]:
ventura_building['type'].unique()

array(['Multi', 'Single'], dtype=object)

In [118]:
single['type'].value_counts()

type
Single    159367778
Multi      38931128
Name: count, dtype: int64

In [ ]:
ventura_building.plot(column = "type")

## Old code:

In [25]:
## FINDING ALL MULTI-FAMILY BUILDINGS BY JOINING ZILLOW -> PARCEL, THEN PARCEL -> BUILDINGS

# select only multi-family data from Zillow
zillow_multi = zillow[zillow['type'] == "Multi"]
zillow_multi = zillow_multi[zillow_multi['code'] != "RR106"]

## crop only to residential parcels
# keep the indices where multi-family homes match to parcels (.index.unique() de-duplicates)
valid_parcels = parcels.sjoin(zillow_multi, how = "inner", predicate="intersects")[parcels.columns].index.unique()

# select the parcels that match these indices
parcels_res = parcels.loc[valid_parcels]

# confirm that joining with Zillow decreases the number of parcels
assert len(parcels_res) < len(parcels)

# crop to residential buildings (by keeping only those within residential parcels)
valid_buildings = building.sjoin(parcels_res, predicate="intersects").index.unique()
buildings_res = building.loc[valid_buildings]

# confirm that joining with Zillow decreased the number of buildings
assert len(buildings_res) < len(building)

## join parcels to buildings (keeping observations as parcels, but with building attributes)
# sum number of units per parcel
#units_sum = parcels_res.sjoin(zillow_multi, predicate="intersects").groupby(level=0)["unit"].sum()

# join on parcels with summed number of units
#parcels_res = parcels_res.join(units_sum)



## CALCULATING VOLUME FOR BOTH SINGLE AND MULTI-FAMILY HOMES

# keep all residential buildings, and add zillow points only where they match up (MULTI)
building_zillow_multi = gpd.sjoin(
    buildings_res,
    zillow_multi,
    how = "left",
    predicate = "intersects")

# filter for only single and condo units
zillow_single = zillow[(zillow['type'] == "Single") | (zillow['code'] == "RR106")]

# keep all residential buildings, and add zillow points only where they match up (SINGLE & CONDOS)
building_zillow_single = gpd.sjoin(
    building,
    zillow_single,
    how = "inner",
    predicate = "intersects")

# combine all for volume calculation
building_zillow_all = pd.concat([building_zillow_multi, building_zillow_single])

# reproject data frame to crs with meters as units
building_m = building_zillow_all.to_crs("EPSG:6933")

# create column from polygon area
building_m['area_m2'] = building_m.geometry.area

# rename height column to be clear about units
building_m.rename(columns={"height":"height_m"}, inplace = True)

# create volume column
building_m['volume_m3'] = building_m['area_m2'] * building_m['height_m']

# save single and condos as its own df
non_multi = building_m[(building_m['type'] == "Single") | (building_m['code'] == "RR106")]

# now that single unit homes also have volume data, we can drop them
building_m = building_m[building_m['type'] == "Multi"]
building_m = building_m[building_m['code'] != "RR106"]



## BUILDING REGRESSION TO PREDICT UNIT DATA WHERE IT IS MISSING

# keep only observations with unit data
building_w_units = building_m[~building_m['unit'].isna()]

assert building_w_units['unit'].isna().sum() == 0

# run model
results = smf.ols('unit ~ volume_m3', data=building_w_units).fit()

# add residuals as a column
building_w_units['residual'] = results.resid.copy()

# keep only observations that are less/equal to 2 standard deviations from residuals
building_units_clean = building_w_units[building_w_units['residual'].abs() <= 2 * building_w_units['residual'].std()]

# save outliers, as we will re-predict them using the regression
building_outliers = building_w_units[building_w_units['residual'].abs() > 2 * building_w_units['residual'].std()]

# rerun linear regression
results_clean = smf.ols('unit ~ volume_m3', data=building_units_clean).fit()

# save variables
intercept = results_clean.params[0]
slope = results_clean.params[1]

# extract just the multi-family homes where unit info is missing
missing_units = building_m[building_m['unit'].isna()]

# combine dataframes with missing unit data as well as outliers (since both will be predicted)
missing_outlier_units = pd.concat([building_outliers, missing_units])

assert len(missing_units) < len(missing_outlier_units)

# replace unit column with prediction
missing_outlier_units_pred = missing_outlier_units.copy().drop('unit', axis = 1)

missing_outlier_units_pred = missing_outlier_units_pred.reset_index(drop=True)

missing_outlier_units_pred['unit'] = round(intercept + missing_outlier_units_pred['volume_m3'] * slope)

# combine multi-family homes data frames
multi_complete = pd.concat([building_w_units, missing_outlier_units_pred]).to_crs(zillow.crs)

# drop excess columns
multi_complete = multi_complete.drop(['residual'], axis = 1)
multi_complete = multi_complete.drop(['index_right'], axis = 1) 

## UNIT PREDICTION COMPLETE



## AGGREGATE BY PARCEL (SUM MULTI-HOME UNITS)

# find the parcels that contain multi-family homes
multi_by_parcel = parcels_res.sjoin(multi_complete, predicate="intersects")

# when aggregated to parcel, there should be less multi-family home observations
#assert len(multi_by_parcel) < len(multi_complete)

# join complete multi-family homes data frame to parcels, then sum units by parcel
summed_units = parcels_res.sjoin(multi_complete, predicate="intersects").groupby(["PARNO"])['unit'].sum()

assert len(summed_units) < len(multi_by_parcel) # this should hold because multi_by_parcel has rows for each building, 
                                                # whereas multi_summed_units is by parcel (and there are less parcels than buildings)

# join unit sums to the parcel geometries themselves, keeping only where units were summed
multi_summed_units = parcels_res.join(summed_units).dropna(subset = ['unit'])

assert len(multi_summed_units) < len(multi_by_parcel)

## SAVING NON-MULTI-FAMILY (SINGLE AND CONDO) OBSERVATIONS

non_multi = non_multi[multi_complete.columns].to_crs(zillow.crs)

# keep only variables of interest
# non_multi = non_multi[['type', 'source', 'id', 'height_m', 'var', 'region', 'bbox', 'geometry', 'area_m2', 'volume_m3']].to_crs(zillow.crs)

# join Zillow data to non-multi family homes (takes ~1 minute)
non_multi_points = gpd.sjoin(
    zillow, # left df's geometry is always kept
    non_multi,
    how = "inner",
    predicate = "intersects")

/Users/sarak/.conda/envs/electrigrid-env/lib/python3.11/site-packages/geopandas/geodataframe.py:1528: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)
/tmp/ipykernel_1589128/2097509032.py:99: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  intercept = results_clean.params[0]
/tmp/ipykernel_1589128/2097509032.py:100: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.il

In [17]:
# concatenate multi and single family homes
ventura_building = pd.concat([non_multi, multi_complete]).to_crs(zillow.crs)

In [ ]:
# non_multi_points_test.to_file("data/non_multi_zillow_test.geojson", driver='GeoJSON')

In [ ]:
# multi_summed_units_test.to_file("data/multi_summed_units_test.geojson", driver='GeoJSON')